# Level 3 — Business Analytics

## Objective

This notebook answers business questions using SQL queries and summarizes insights for stakeholders.

In [1]:
import pandas as pd
import duckdb

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

con = duckdb.connect()

con.execute("""
CREATE OR REPLACE VIEW superstore_features AS
SELECT *
FROM read_csv_auto('../data/superstore_features.csv');
""")

con.execute("""
SELECT *
FROM superstore_features
LIMIT 5
""").df()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit,fulfillment_days,profit_margin,order_year,order_month,customer_lifetime_sales,customer_span_days,customer_tier
0,2230,CA-2014-128055,2014-03-31,2014-04-05,Standard Class,AA-10315,Alex Avila,Consumer,United States,San Francisco,California,94122,West,OFF-BI-10004390,Office Supplies,Binders,GBC DocuBind 200 Manual Binding Machine,673.568,2,0.2,252.5880,5,0.3750,2014,3,5563.56,1186,Premium
1,2231,CA-2014-128055,2014-03-31,2014-04-05,Standard Class,AA-10315,Alex Avila,Consumer,United States,San Francisco,California,94122,West,OFF-AP-10002765,Office Supplies,Appliances,Fellowes Advanced Computer Series Surge Protec...,52.980,2,0.0,14.8344,5,0.2800,2014,3,5563.56,1186,Premium
2,5199,CA-2016-103982,2016-03-03,2016-03-08,Standard Class,AA-10315,Alex Avila,Consumer,United States,Round Rock,Texas,78664,Central,OFF-SU-10000151,Office Supplies,Supplies,High Speed Automatic Electric Letter Opener,3930.072,3,0.2,-786.0144,5,-0.2000,2016,3,5563.56,1186,Premium
3,5200,CA-2016-103982,2016-03-03,2016-03-08,Standard Class,AA-10315,Alex Avila,Consumer,United States,Round Rock,Texas,78664,Central,OFF-FA-10001332,Office Supplies,Fasteners,"Acco Banker's Clasps, 5 3/4""-Long",2.304,1,0.2,0.7776,5,0.3375,2016,3,5563.56,1186,Premium
4,5201,CA-2016-103982,2016-03-03,2016-03-08,Standard Class,AA-10315,Alex Avila,Consumer,United States,Round Rock,Texas,78664,Central,TEC-PH-10000895,Technology,Phones,Polycom VVX 310 VoIP phone,431.976,3,0.2,32.3982,5,0.0750,2016,3,5563.56,1186,Premium


## Selecting Variables for Business Analysis

The engineered dataset contains both the original retail variables and the features created in **Notebook 02**. While all variables remain available in the exported dataset, only those relevant to the business analyses in this notebook were selected.

The following columns were excluded for the reasons below:

- **`row_id`** was excluded because it serves only as a unique row identifier and does not provide meaningful information for the planned business analyses.

- **`order_date`** was excluded because the engineered features **`order_year`** and **`order_month`** provide the time granularity needed for this analysis.

- **`ship_date`** was excluded because the engineered feature **`fulfillment_days`** more directly measures shipping performance.

- **`customer_id`** was excluded in favor of **`customer_name`**, which produces more interpretable customer-level reports.

- **`product_id`** and **`product_name`** were excluded from product-level analysis because Notebook 01 identified inconsistencies in the relationship between these fields. Without sufficient information to determine the correct identifier-name relationships, using either field to make individual-product comparisons could produce unreliable conclusions. Product performance is therefore evaluated at the independently validated **`category`** and **`sub_category`** levels.

- **`country`** was excluded because every observation occurred in the United States, providing no additional analytical value.

- **`city`** and **`postal_code`** were excluded because they contain **531** and **631** unique values, respectively. For this analysis, **`region`** and **`state`** provide a more meaningful level of geographic aggregation while reducing unnecessary granularity.

The resulting dataset retains the variables most relevant to analyzing customer behavior, category and subcategory performance, geographic trends, profitability, and operational efficiency. This selection also ensures that the analyses rely only on fields whose relationships were validated during preprocessing, keeping the findings focused, interpretable, and defensible.



In [2]:
con.execute("""
CREATE OR REPLACE VIEW analysis_data AS

SELECT
    -- Customer
    customer_name,
    segment,
    customer_lifetime_sales,  -- Engineered
    customer_span_days,       -- Engineered
    customer_tier,            -- Engineered

    -- Order and Time
    order_id,
    order_year,               -- Engineered
    order_month,              -- Engineered

    -- Geography
    region,
    state,

    -- Product Classification
    category,
    sub_category,

    -- Shipping
    ship_mode,
    fulfillment_days,         -- Engineered

    -- Sales and Profitability
    sales,
    quantity,
    discount,
    profit,
    profit_margin             -- Engineered

FROM superstore_features;
""")

con.execute("""
SELECT *
FROM analysis_data
LIMIT 5;
""").df()


,customer_name,segment,order_id,order_date,region,state,category,sub_category,ship_mode,sales,quantity,discount,profit,order_year,order_month,fulfillment_days,profit_margin,customer_lifetime_sales,customer_span_days,customer_tier
0,Alex Avila,Consumer,CA-2014-128055,2014-03-31,West,California,Office Supplies,Binders,Standard Class,673.568,2,0.2,252.5880,2014,3,5,0.3750,5563.56,1186,Premium
1,Alex Avila,Consumer,CA-2014-128055,2014-03-31,West,California,Office Supplies,Appliances,Standard Class,52.980,2,0.0,14.8344,2014,3,5,0.2800,5563.56,1186,Premium
2,Alex Avila,Consumer,CA-2016-103982,2016-03-03,Central,Texas,Office Supplies,Supplies,Standard Class,3930.072,3,0.2,-786.0144,2016,3,5,-0.2000,5563.56,1186,Premium
3,Alex Avila,Consumer,CA-2016-103982,2016-03-03,Central,Texas,Office Supplies,Fasteners,Standard Class,2.304,1,0.2,0.7776,2016,3,5,0.3375,5563.56,1186,Premium
4,Alex Avila,Consumer,CA-2016-103982,2016-03-03,Central,Texas,Technology,Phones,Standard Class,431.976,3,0.2,32.3982,2016,3,5,0.0750,5563.56,1186,Premium


## Customer Analysis

Customer analytics focuses on understanding purchasing behavior across individual customers and customer segments. By identifying high-value customers, comparing segment performance, and evaluating customer lifetime sales, we can better understand who generates the greatest value for the business and where customer retention efforts should be focused.

### Top Customers by Lifetime Sales

**Question:**

Who are the ten highest-value customers based on lifetime sales, and what do their ordering frequency, average spending per order, and observed customer span reveal about their purchasing behavior?

**Objective:**

Identify the ten customers with the highest cumulative sales and compare their total number of orders, average spending per order, and the number of days between their first and most recent recorded purchases. This analysis helps determine whether high customer value is driven primarily by frequent purchasing, larger orders, a longer purchasing history, or a combination of these factors.

**Variables:**

- `customer_name`
- `customer_lifetime_sales`
- `order_id`
- `sales`
- `customer_span_days`

In [3]:
con.execute("""
SELECT
    customer_name,
    ANY_VALUE(customer_lifetime_sales) AS customer_lifetime_sales,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(sales) / COUNT(DISTINCT order_id) AS avg_spend_per_order,
    ANY_VALUE(customer_span_days) AS customer_span_days,

    365.25 * SUM(sales) /
        NULLIF(ANY_VALUE(customer_span_days), 0)
        AS annualized_spending_rate

FROM analysis_data

GROUP BY customer_name

ORDER BY customer_lifetime_sales DESC

LIMIT 10;
""").df().style.hide(axis="index").format({
    "customer_lifetime_sales": "${:,.2f}",
    "total_orders": "{:,}",
    "avg_spend_per_order": "${:,.2f}",
    "customer_span_days": "{:,}",
    "annualized_spending_rate": "${:,.2f}"
})

customer_name,customer_lifetime_sales,total_orders,avg_spend_per_order,customer_span_days,annualized_spending_rate
Sean Miller,"$25,043.05",5,"$5,008.61","1,304","$7,014.55"
Tamara Chand,"$19,052.22",5,"$3,810.44",750,"$9,278.43"
Raymond Buch,"$15,117.34",6,"$2,519.56",542,"$10,187.47"
Tom Ashbrook,"$14,595.62",4,"$3,648.91","1,136","$4,692.83"
Adrian Barton,"$14,473.57",10,"$1,447.36","1,065","$4,963.82"
Ken Lonsdale,"$14,175.23",12,"$1,181.27","1,207","$4,289.56"
Sanjit Chand,"$14,142.33",9,"$1,571.37","1,068","$4,836.60"
Hunter Lopez,"$12,873.30",6,"$2,145.55","1,397","$3,365.76"
Sanjit Engle,"$12,209.44",11,"$1,109.95","1,350","$3,303.33"
Christopher Conant,"$12,129.07",5,"$2,425.81",543,"$8,158.64"


**Results:**

Sean Miller was the highest-value customer, generating **$25,043.05** across 5 orders, with an average spend of **$5,008.61 per order** and an observed purchasing span of **1,304 days**. Tamara Chand ranked second with **$19,052.22**, followed by Raymond Buch with **$15,117.34**.

Although Sean Miller generated the greatest lifetime sales, Raymond Buch had the highest observed annualized spending rate at **$10,187.47 per year**. Tamara Chand ranked second by this measure at **$9,278.43 per year**, followed by Christopher Conant at **$8,158.64 per year**.

**Key Insights:**

High customer value was produced through several different purchasing patterns. Sean Miller accumulated the most lifetime sales through relatively few, exceptionally large orders, averaging **$5,008.61 per order**. Ken Lonsdale followed a frequency-driven pattern, placing the most orders among the top ten but spending substantially less per order.

The annualized spending rate provides additional context by accounting for each customer's observed purchasing span. Raymond Buch accumulated sales at the fastest annualized rate despite ranking third in lifetime sales, while Hunter Lopez accumulated sales more slowly across the longest observed span. This demonstrates that lifetime sales alone can conceal meaningful differences in purchasing frequency, order size, and the pace at which customer value develops.


**Methodological Note:**

The annualized spending rate represents the rate at which recorded sales accumulated between a customer's first and most recent orders. It is a descriptive measure based on observed purchasing history and should not be interpreted as a forecast of future annual spending.

### Sales Performance by Customer Segment

**Question:**

How do revenue, profitability, customer count, and average sales per customer differ across customer segments?

**Objective:**

Compare total sales, total profit, profit margin, customer count, and average sales per customer across the Consumer, Corporate, and Home Office segments. This analysis identifies whether each segment's financial contribution is driven primarily by the size of its customer base, higher spending per customer, stronger profitability, or a combination of these factors.

**Variables:**

- `segment`
- `sales`
- `profit`
- `customer_name`

In [28]:
con.execute("""
SELECT
    segment,

    SUM(sales) AS total_sales,

    SUM(profit) AS total_profit,

    100.0 * SUM(sales) / SUM(SUM(sales)) OVER ()
        AS percentage_of_total_sales,

    100.0 * SUM(profit) / SUM(sales)
        AS profit_margin_percentage,

    COUNT(DISTINCT customer_name) AS customer_count,

    SUM(sales) / COUNT(DISTINCT customer_name)
        AS avg_sales_per_customer


FROM analysis_data
GROUP BY segment
ORDER BY total_sales DESC;
""").df().style.hide(axis="index").format({
    "customer_count": "{:,}",
    "total_sales": "${:,.2f}",
    "percentage_of_total_sales": "{:.2f}%",
    "avg_sales_per_customer": "${:,.2f}",
    "total_profit": "${:,.2f}",
    "profit_margin_percentage": "{:.2f}%"
})

segment,total_sales,total_profit,percentage_of_total_sales,profit_margin_percentage,customer_count,avg_sales_per_customer
Consumer,"$1,161,401.34","$134,119.21",50.56%,11.55%,409,"$2,839.61"
Corporate,"$706,146.37","$91,979.13",30.74%,13.03%,236,"$2,992.15"
Home Office,"$429,653.15","$60,298.68",18.70%,14.03%,148,"$2,903.06"


**Results:**

The Consumer segment generated the highest total sales at **$1,161,401.34** and the highest total profit at **$134,119.21**. It represented **50.56%** of company sales and contained the largest customer base at **409 customers**.

Corporate generated **$706,146.37** in sales and **$91,979.13** in profit across **236 customers**. It achieved the highest average sales per customer at **$2,992.15**. Home Office was the smallest segment, with **148 customers** and **$429,653.15** in sales, but produced the highest profit margin at **14.03%**.

**Key Insights:**

Consumer was the largest contributor to overall revenue and profit, primarily because it contained the greatest number of customers. However, its **11.55% profit margin** was the lowest of the three segments, indicating that its larger revenue base did not translate into the strongest profitability relative to sales.

Corporate customers generated the highest average sales per customer, while Home Office achieved the strongest profit margin despite producing the lowest total sales. These results demonstrate that the smallest segment was not necessarily the least valuable. Consumer may offer the greatest opportunity for margin improvement, while Corporate and Home Office demonstrate stronger per-customer or profitability performance.

### Sales and Profitability by Customer Tier

**Question:**

How do overall financial contribution and average customer profitability differ across customer tiers?

**Objective:**

Compare customer count, total sales, total profit, average sales per customer, average profit per customer, and overall profit margin across customer tiers. This analysis evaluates whether higher-spending customer tiers also generate stronger profitability while accounting for differences in tier size.

**Variables:**

- `customer_tier`
- `customer_name`
- `sales`
- `profit`

In [5]:
con.execute("""
SELECT
    customer_tier,
    COUNT(DISTINCT customer_name) AS customer_count,
    SUM(sales) AS total_sales,
    SUM(profit) AS total_profit,

    SUM(sales) / COUNT(DISTINCT customer_name)
        AS avg_sales_per_customer,

    SUM(profit) / COUNT(DISTINCT customer_name)
        AS avg_profit_per_customer,

    100.0 * SUM(profit) / SUM(sales)
        AS profit_margin_percentage

FROM analysis_data

GROUP BY customer_tier

ORDER BY avg_sales_per_customer DESC;
""").df().style.hide(axis="index").format({
    "customer_count": "{:,}",
    "total_sales": "${:,.2f}",
    "total_profit": "${:,.2f}",
    "avg_sales_per_customer": "${:,.2f}",
    "avg_profit_per_customer": "${:,.2f}",
    "profit_margin_percentage": "{:.2f}%"
})

customer_tier,customer_count,total_sales,total_profit,avg_sales_per_customer,avg_profit_per_customer,profit_margin_percentage
Elite,80,"$709,342.71","$122,167.79","$8,866.78","$1,527.10",17.22%
Premium,119,"$557,107.99","$65,687.41","$4,681.58",$552.00,11.79%
High Value,198,"$574,885.76","$52,925.88","$2,903.46",$267.30,9.21%
Standard,396,"$455,864.40","$45,615.94","$1,151.17",$115.19,10.01%


**Results:**

The Elite tier contained only **80 customers** but generated the highest total sales at **$709,342.71** and the highest total profit at **$122,167.79**. It also led all tiers in average sales per customer at **$8,866.78**, average profit per customer at **$1,527.10**, and overall profit margin at **17.22%**.

Premium customers averaged **$4,681.58** in sales and **$552.00** in profit per customer. Although the High Value tier generated slightly more total sales than Premium—**$574,885.76** compared with **$557,107.99**—it contained 79 more customers and produced lower average sales, average profit, and profit margin. The Standard tier contained the most customers at **396**, but had the lowest average sales and profit per customer.

**Key Insights:**

Elite customers are disproportionately valuable. Despite representing approximately **10% of the customer base**, they contributed about **30.88% of total sales** and **42.66% of total profit**. Their strong performance across both per-customer measures and profit margin indicates that their value is not simply caused by the tier's size.

The comparison between Premium and High Value customers demonstrates why totals should be interpreted alongside customer counts. High Value generated slightly more total sales because it contained more customers, while the average Premium customer generated substantially greater sales and more than twice as much profit. The High Value tier also had the lowest profit margin at **9.21%**, suggesting that discounting, product mix, or other cost drivers may warrant further investigation. Retention efforts should prioritize Elite and Premium customers while profitability improvement efforts may be especially valuable within the High Value tier.

## Geographic Analysis

Geographic analysis evaluates business performance across regions and states. Comparing sales, profitability, and customer activity by location helps identify high-performing markets, uncover regional trends, and highlight areas that may benefit from targeted business strategies.

### Financial Performance by Region

**Question:**

Which regions generate the greatest financial value when considering sales, profit, profit margin, and average order value?

**Objective:**

Compare total orders, sales, profit, profit margin, and average order value across regions. This analysis determines whether regional performance is driven by greater order volume, higher-value orders, stronger profitability, or a combination of these factors.

**Variables:**

- `region`
- `order_id`
- `sales`
- `profit`

In [10]:
con.execute("""
SELECT
    region,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(sales) AS total_sales,
    SUM(profit) AS total_profit,

    100.0 * SUM(sales) / SUM(SUM(sales)) OVER ()
        AS percentage_of_total_sales,

    SUM(sales) / COUNT(DISTINCT order_id)
        AS avg_order_value,

    100.0 * SUM(profit) / SUM(sales)
        AS profit_margin_percentage

FROM analysis_data

GROUP BY region

ORDER BY total_sales DESC;
""").df().style.hide(axis="index").format({
    "total_orders": "{:,}",
    "total_sales": "${:,.2f}",
    "total_profit": "${:,.2f}",
    "percentage_of_total_sales": "{:.2f}%",
    "avg_order_value": "${:,.2f}",
    "profit_margin_percentage": "{:.2f}%"
})

region,total_orders,total_sales,total_profit,percentage_of_total_sales,avg_order_value,profit_margin_percentage
West,"1,611","$725,457.82","$108,418.45",31.58%,$450.32,14.94%
East,"1,401","$678,781.24","$91,522.78",29.55%,$484.50,13.48%
Central,"1,175","$501,239.89","$39,706.36",21.82%,$426.59,7.92%
South,822,"$391,721.91","$46,749.43",17.05%,$476.55,11.93%


**Results:**

The West was the strongest-performing region, generating **$725,457.82** in sales and **$108,418.45** in profit across 1,611 orders. It contributed **31.58%** of total sales and achieved the highest regional profit margin at **14.94%**.

The East ranked second in total sales at **$678,781.24** and had the highest average order value at **$484.50**. The Central region generated **$501,239.89** in sales but produced only **$39,706.36** in profit, resulting in the lowest regional profit margin at **7.92%**. The South generated the lowest total sales but had a relatively strong average order value of **$476.55**.

**Key Insights:**

The West's leading performance was supported by a combination of the greatest order volume, highest total sales, highest total profit, and strongest profit margin. The East also performed strongly, generating the highest average order value and the second-highest profit margin.

The Central region warrants further investigation because its **7.92% profit margin** was substantially below the other regions despite generating more than half a million dollars in sales. This suggests that revenue volume alone did not translate into equally strong financial returns. In contrast, the South's relatively high average order value and **11.93% margin** indicate that its lower total contribution was driven more by limited order volume than weak order economics.

### Highest-Performing States

**Question:**

Which ten states generate the highest total sales, and how do their profitability and average order values compare?

**Objective:**

Identify the ten states with the highest sales and compare their total orders, total profit, average order value, and profit margin. This analysis determines whether the largest state markets also generate strong financial returns.

**Variables:**

- `state`
- `region`
- `order_id`
- `sales`
- `profit`

In [11]:
con.execute("""
SELECT
    state,
    region,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(sales) AS total_sales,
    SUM(profit) AS total_profit,

    SUM(sales) / COUNT(DISTINCT order_id)
        AS avg_order_value,

    100.0 * SUM(profit) / SUM(sales)
        AS profit_margin_percentage

FROM analysis_data

GROUP BY state, region

ORDER BY total_sales DESC

LIMIT 10;
""").df().style.hide(axis="index").format({
    "total_orders": "{:,}",
    "total_sales": "${:,.2f}",
    "total_profit": "${:,.2f}",
    "avg_order_value": "${:,.2f}",
    "profit_margin_percentage": "{:.2f}%"
})

state,region,total_orders,total_sales,total_profit,avg_order_value,profit_margin_percentage
California,West,"1,021","$457,687.63","$76,381.39",$448.27,16.69%
New York,East,562,"$310,876.27","$74,038.55",$553.16,23.82%
Texas,Central,487,"$170,188.05","$-25,729.36",$349.46,-15.12%
Washington,West,256,"$138,641.27","$33,402.65",$541.57,24.09%
Pennsylvania,East,288,"$116,511.91","$-15,559.96",$404.56,-13.35%
Florida,South,200,"$89,473.71","$-3,399.30",$447.37,-3.80%
Illinois,Central,276,"$80,166.10","$-12,607.89",$290.46,-15.73%
Ohio,East,236,"$78,258.14","$-16,971.38",$331.60,-21.69%
Michigan,Central,117,"$76,269.61","$24,463.19",$651.88,32.07%
Virginia,South,115,"$70,636.72","$18,597.95",$614.23,26.33%


**Results:**

California generated the highest state-level sales at **$457,687.63** and the highest total profit at **$76,381.39**. New York ranked second in sales at **$310,876.27** and produced nearly as much profit as California—**$74,038.55**—despite recording 459 fewer orders. New York's profit margin was **23.82%**, compared with California's **16.69%**.

Several high-revenue states were not profitable. Texas ranked third in sales at **$170,188.05** but generated a **$25,729.36 loss**. Pennsylvania, Florida, Illinois, and Ohio also appeared among the ten highest-sales states while producing negative total profit. Michigan had the highest average order value at **$651.88** and the strongest profit margin at **32.07%** among the ten states shown.

**Key Insights:**

High sales did not consistently indicate strong financial performance. California and New York combined substantial revenue with positive profit, while Texas generated the third-highest sales but the largest state-level loss. Five of the ten highest-sales states produced negative profit, demonstrating that market size should not be evaluated independently of profitability.

New York was particularly efficient, producing nearly as much total profit as California from substantially fewer orders. Michigan and Virginia also produced strong average order values and profit margins despite their lower sales totals. These states may represent financially efficient markets with potential for carefully targeted growth, while Texas, Ohio, Pennsylvania, Illinois, and Florida require investigation into the factors reducing profitability.

### States Generating Financial Losses

**Question:**

Which states generated an overall financial loss despite producing sales?

**Objective:**

Identify states with negative total profit and compare their sales, order volume, average line-level discount, and profit margin. This analysis highlights geographic markets where revenue is not translating into positive financial returns and provides an initial indication of whether discounting may warrant further investigation.

**Variables:**

- `state`
- `region`
- `order_id`
- `sales`
- `profit`
- `discount`

In [12]:
con.execute("""
SELECT
    state,
    region,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(sales) AS total_sales,
    SUM(profit) AS total_profit,

    100.0 * AVG(discount)
        AS avg_line_discount_percentage,

    100.0 * SUM(profit) / SUM(sales)
        AS profit_margin_percentage

FROM analysis_data

GROUP BY state, region

HAVING SUM(profit) < 0

ORDER BY total_profit;
""").df().style.hide(axis="index").format({
    "total_orders": "{:,}",
    "total_sales": "${:,.2f}",
    "total_profit": "${:,.2f}",
    "avg_line_discount_percentage": "{:.2f}%",
    "profit_margin_percentage": "{:.2f}%"
})

state,region,total_orders,total_sales,total_profit,avg_line_discount_percentage,profit_margin_percentage
Texas,Central,487,"$170,188.05","$-25,729.36",37.02%,-15.12%
Ohio,East,236,"$78,258.14","$-16,971.38",32.49%,-21.69%
Pennsylvania,East,288,"$116,511.91","$-15,559.96",32.86%,-13.35%
Illinois,Central,276,"$80,166.10","$-12,607.89",39.00%,-15.73%
North Carolina,South,136,"$55,603.16","$-7,490.91",28.35%,-13.47%
Colorado,West,79,"$32,108.12","$-6,527.86",31.65%,-20.33%
Tennessee,South,91,"$30,661.87","$-5,341.69",29.13%,-17.42%
Arizona,West,108,"$35,282.00","$-3,427.92",30.36%,-9.72%
Florida,South,200,"$89,473.71","$-3,399.30",29.93%,-3.80%
Oregon,West,56,"$17,431.15","$-1,190.47",28.87%,-6.83%


**Results:**

Ten states generated an overall financial loss. Texas produced the largest loss at **$25,729.36**, despite generating **$170,188.05** in sales across 487 orders. Ohio had the most negative profit margin at **-21.69%**, followed by Colorado at **-20.33%** and Tennessee at **-17.42%**.

Illinois recorded the highest average line-level discount among the loss-generating states at **39.00%**, while Texas averaged **37.02%**. Across all ten unprofitable states, average line-level discounts ranged from **28.35% to 39.00%**.

**Key Insights:**

The loss-generating states produced meaningful revenue, but their sales did not translate into positive financial returns. Texas is the most significant concern because it combined the highest sales and order volume among the unprofitable states with the largest total loss. Ohio and Colorado also warrant attention because more than 20% of their sales revenue was lost based on their overall profit margins.

The consistently elevated average discounts among these states suggest that discounting may be associated with weak geographic profitability. However, this table alone does not establish that discounts caused the losses because it does not compare discount patterns with profitable states or control for product mix. The relationship between discounting and profitability should therefore be evaluated directly in the Sales and Profitability Analysis section.

## Product Classification Analysis

Product classification analysis examines sales and profitability across the validated `category` and `sub_category` levels. The objective is to identify high-performing classifications, recognize areas generating financial losses, and understand which parts of the product portfolio contribute most to overall business success. Individual products are not analyzed because Notebook 01 identified inconsistencies between `product_id` and `product_name`.

### Financial Performance by Product Category

**Question:**

Which product categories generate the greatest sales and profit, and how do their profit margins compare?

**Objective:**

Compare order activity, units sold, total sales, total profit, contribution to company sales, and overall profit margin across product categories. This analysis determines whether the highest-revenue categories also produce the strongest financial returns.

**Variables:**

- `category`
- `order_id`
- `quantity`
- `sales`
- `profit`

In [13]:
con.execute("""
SELECT
    category,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(quantity) AS total_units_sold,
    SUM(sales) AS total_sales,
    SUM(profit) AS total_profit,

    100.0 * SUM(sales) / SUM(SUM(sales)) OVER ()
        AS percentage_of_total_sales,

    100.0 * SUM(profit) / SUM(sales)
        AS profit_margin_percentage

FROM analysis_data

GROUP BY category

ORDER BY total_sales DESC;
""").df().style.hide(axis="index").format({
    "total_orders": "{:,}",
    "total_units_sold": "{:,}",
    "total_sales": "${:,.2f}",
    "total_profit": "${:,.2f}",
    "percentage_of_total_sales": "{:.2f}%",
    "profit_margin_percentage": "{:.2f}%"
})

category,total_orders,total_units_sold,total_sales,total_profit,percentage_of_total_sales,profit_margin_percentage
Technology,"1,544","6,939.0","$836,154.03","$145,454.95",36.40%,17.40%
Furniture,"1,764","8,028.0","$741,999.80","$18,451.27",32.30%,2.49%
Office Supplies,"3,742","22,906.0","$719,047.03","$122,490.80",31.30%,17.04%


**Results:**

Technology generated the highest total sales at **$836,154.03**, representing **36.40%** of company sales. It also produced the highest total profit at **$145,454.95** and the strongest category-level profit margin at **17.40%**.

Furniture ranked second in sales at **$741,999.80**, but generated only **$18,451.27** in profit and had a margin of just **2.49%**. Office Supplies generated the lowest category sales at **$719,047.03**, but produced **$122,490.80** in profit with a strong **17.04%** margin. It also recorded the greatest activity, with 3,742 orders and 22,906 units sold.

**Key Insights:**

Technology was the strongest category overall, leading in sales, total profit, and profit margin. Office Supplies also performed efficiently: despite generating the lowest sales of the three categories, it produced nearly as much profit as Technology and achieved a comparable profit margin.

Furniture's financial performance was substantially weaker. Although it generated more sales than Office Supplies, its total profit was approximately **$104,040 lower**, and its **2.49% margin** indicates that very little of its revenue was retained as profit. This demonstrates that strong revenue and unit volume do not necessarily translate into strong financial value. Furniture should therefore receive further investigation at the subcategory level.

### Financial Performance by Product Subcategory

**Question:**

Which product subcategories generate the highest sales and profit, and are the highest-selling subcategories also the most profitable?

**Objective:**

Compare total orders, units sold, sales, profit, and profit margin across product subcategories. This analysis identifies the classifications that contribute the most financial value and highlights differences between revenue performance and profitability.

**Variables:**

- `category`
- `sub_category`
- `order_id`
- `quantity`
- `sales`
- `profit`

In [14]:
con.execute("""
SELECT
    category,
    sub_category,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(quantity) AS total_units_sold,
    SUM(sales) AS total_sales,
    SUM(profit) AS total_profit,

    100.0 * SUM(profit) / SUM(sales)
        AS profit_margin_percentage

FROM analysis_data

GROUP BY category, sub_category

ORDER BY total_sales DESC;
""").df().style.hide(axis="index").format({
    "total_orders": "{:,}",
    "total_units_sold": "{:,}",
    "total_sales": "${:,.2f}",
    "total_profit": "${:,.2f}",
    "profit_margin_percentage": "{:.2f}%"
})

category,sub_category,total_orders,total_units_sold,total_sales,total_profit,profit_margin_percentage
Technology,Phones,814,"3,289.0","$330,007.05","$44,515.73",13.49%
Furniture,Chairs,576,"2,356.0","$328,449.10","$26,590.17",8.10%
Office Supplies,Storage,777,"3,158.0","$223,843.61","$21,278.83",9.51%
Furniture,Tables,307,"1,241.0","$206,965.53","$-17,725.48",-8.56%
Office Supplies,Binders,"1,316","5,974.0","$203,412.73","$30,221.76",14.86%
Technology,Machines,112,440.0,"$189,238.63","$3,384.76",1.79%
Technology,Accessories,718,"2,976.0","$167,380.32","$41,936.64",25.05%
Technology,Copiers,68,234.0,"$149,528.03","$55,617.82",37.20%
Furniture,Bookcases,224,868.0,"$114,880.00","$-3,472.56",-3.02%
Office Supplies,Appliances,451,"1,729.0","$107,532.16","$18,138.01",16.87%


**Results:**

Phones generated the highest subcategory sales at **$330,007.05** and produced **$44,515.73** in profit. Chairs ranked closely behind with **$328,449.10** in sales, but generated considerably less profit at **$26,590.17** and achieved a lower margin of **8.10%**.

Copiers produced the highest total profit at **$55,617.82** and a strong **37.20% profit margin**, despite appearing in only 68 orders. Paper and Labels achieved the highest profit margins at **43.39%** and **44.42%**, respectively, although their total sales were considerably lower. Tables ranked fourth in sales at **$206,965.53** but generated a **$17,725.48 loss**.

**Key Insights:**

The highest-selling subcategories were not always the most profitable. Phones combined high sales with substantial profit, while Chairs generated nearly identical sales but produced approximately **$17,926 less profit**. Copiers demonstrated particularly strong financial efficiency by generating the highest subcategory profit from relatively few orders and units.

Several lower-revenue Office Supplies subcategories, including Paper and Labels, produced exceptionally strong margins. In contrast, Tables generated substantial revenue but operated at a loss. These differences show why subcategory performance should be evaluated using both total contribution and profit margin rather than sales rankings alone.

### Subcategories Generating Financial Losses

**Question:**

Which product subcategories generated an overall financial loss, and what discount levels were associated with their performance?

**Objective:**

Identify subcategories with negative total profit and compare their order activity, units sold, total sales, average line-level discount, and overall profit margin. This analysis highlights product classifications where meaningful sales activity did not translate into positive financial returns.

**Variables:**

- `category`
- `sub_category`
- `order_id`
- `quantity`
- `sales`
- `profit`
- `discount`

In [15]:
con.execute("""
SELECT
    category,
    sub_category,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(quantity) AS total_units_sold,
    SUM(sales) AS total_sales,
    SUM(profit) AS total_profit,

    100.0 * AVG(discount)
        AS avg_line_discount_percentage,

    100.0 * SUM(profit) / SUM(sales)
        AS profit_margin_percentage

FROM analysis_data

GROUP BY category, sub_category

HAVING SUM(profit) < 0

ORDER BY total_profit;
""").df().style.hide(axis="index").format({
    "total_orders": "{:,}",
    "total_units_sold": "{:,}",
    "total_sales": "${:,.2f}",
    "total_profit": "${:,.2f}",
    "avg_line_discount_percentage": "{:.2f}%",
    "profit_margin_percentage": "{:.2f}%"
})

category,sub_category,total_orders,total_units_sold,total_sales,total_profit,avg_line_discount_percentage,profit_margin_percentage
Furniture,Tables,307,"1,241.0","$206,965.53","$-17,725.48",26.13%,-8.56%
Furniture,Bookcases,224,868.0,"$114,880.00","$-3,472.56",21.11%,-3.02%
Office Supplies,Supplies,187,647.0,"$46,673.54","$-1,189.10",7.68%,-2.55%


**Results:**

Three subcategories generated an overall financial loss. Tables produced the largest loss at **$17,725.48**, despite generating **$206,965.53** in sales across 307 orders. It had an average line-level discount of **26.13%** and an overall profit margin of **-8.56%**.

Bookcases generated **$114,880.00** in sales but lost **$3,472.56**, resulting in a **-3.02% margin**. Supplies generated the smallest loss at **$1,189.10** on **$46,673.54** in sales and had the lowest average line-level discount of the three at **7.68%**.

**Key Insights:**

Furniture accounted for two of the three loss-generating subcategories. Tables and Bookcases together lost approximately **$21,198**, explaining much of the Furniture category's weak **2.49% profit margin**. Tables are the most significant concern because they combined substantial sales with the largest loss and most negative margin.

Tables and Bookcases had higher average discounts than Supplies, suggesting that discounting may contribute to their weak profitability. However, Supplies generated a loss despite a much lower average discount, indicating that discounting alone does not explain every unprofitable subcategory. Additional analysis of discount levels and other product-classification factors is needed before drawing a causal conclusion.

## Sales & Profitability Analysis

Sales and profitability analysis investigates the financial performance of the business by examining revenue, discounts, profit, and profit margins. This section explores how pricing and discounting strategies influence profitability and identifies opportunities to improve financial performance.

### Profitability by Discount Level

**Question:**

How do sales, profit margin, and the frequency of financial losses change across discount levels?

**Objective:**

Compare sales-line activity, units sold, total sales, total profit, profit margin, and the percentage of loss-generating sales lines at each discount level. This analysis identifies the discount levels associated with declining profitability and determines whether larger discounts consistently produce weaker financial results.

**Variables:**

- `discount`
- `quantity`
- `sales`
- `profit`

In [16]:
con.execute("""
SELECT
    100.0 * discount AS discount_percentage,
    COUNT(*) AS sales_line_count,
    SUM(quantity) AS total_units_sold,
    SUM(sales) AS total_sales,
    SUM(profit) AS total_profit,

    100.0 * SUM(profit) / SUM(sales)
        AS profit_margin_percentage,

    100.0 * COUNT(*) FILTER (
        WHERE profit < 0
    ) / COUNT(*) AS loss_line_percentage

FROM analysis_data

GROUP BY discount

ORDER BY discount;
""").df().style.hide(axis="index").format({
    "discount_percentage": "{:.0f}%",
    "sales_line_count": "{:,}",
    "total_units_sold": "{:,.0f}",
    "total_sales": "${:,.2f}",
    "total_profit": "${:,.2f}",
    "profit_margin_percentage": "{:.2f}%",
    "loss_line_percentage": "{:.2f}%"
})

discount_percentage,sales_line_count,total_units_sold,total_sales,total_profit,profit_margin_percentage,loss_line_percentage
0%,"4,798","18,267","$1,087,908.47","$320,987.60",29.51%,0.00%
10%,94,373,"$54,369.35","$9,029.18",16.61%,4.26%
15%,52,198,"$27,558.52","$1,418.99",5.15%,32.69%
20%,"3,657","13,660","$764,594.37","$90,337.31",11.82%,13.73%
30%,227,849,"$103,226.65","$-10,369.28",-10.05%,91.63%
32%,27,105,"$14,493.46","$-2,391.14",-16.50%,100.00%
40%,206,786,"$116,417.78","$-23,057.05",-19.81%,87.38%
45%,11,45,"$5,484.97","$-2,493.11",-45.45%,100.00%
50%,66,241,"$58,918.54","$-20,506.43",-34.80%,100.00%
60%,138,501,"$6,644.70","$-5,944.66",-89.46%,100.00%


**Results:**

Sales lines without a discount generated **$1,087,908.47** in sales and **$320,987.60** in profit, achieving the strongest profit margin at **29.51%** with no loss-generating lines. Discounts of 10%, 15%, and 20% remained profitable overall, although their margins were lower.

Profitability became negative at the 30% discount level, which generated a **$10,369.28 loss** and had a **-10.05% margin**. At this discount, **91.63%** of sales lines resulted in a loss. Every sales line discounted by 45% or more was unprofitable. The 70% discount level generated the largest total loss at **$40,075.36**, while the 80% level produced the most negative margin at **-180.03%**.

**Key Insights:**

The results demonstrate a strong negative relationship between discount size and profitability. Undiscounted sales produced the greatest total profit and strongest margin, while aggregate profitability turned negative beginning at the 30% discount level. Discounts of 30% or more collectively generated substantial financial losses.

The 20% discount level remained profitable overall and supported considerable sales volume, suggesting that moderate discounting may be sustainable in some circumstances. However, the increase in loss-generating lines at 15% and above indicates that the effect of a discount also depends on factors such as product category and underlying margin. Discounts of 30% or more should receive particular scrutiny, while discounts of 45% or more appear financially unsustainable in this dataset.

### Financial Contribution by Profitability Status

**Question:**

What proportion of sales activity and revenue comes from profitable, break-even, and loss-generating sales lines?

**Objective:**

Classify individual sales lines by profitability status and compare their frequency, share of all sales lines, total sales, share of company revenue, and total profit. This analysis measures how much business activity produces positive financial returns and how much revenue is associated with losses.

**Variables:**

- `sales`
- `profit`

In [17]:
con.execute("""
WITH profitability_status AS (
    SELECT
        *,
        CASE
            WHEN profit > 0 THEN 'Profitable'
            WHEN profit = 0 THEN 'Break Even'
            ELSE 'Loss'
        END AS profit_status

    FROM analysis_data
)

SELECT
    profit_status,
    COUNT(*) AS sales_line_count,

    100.0 * COUNT(*) / SUM(COUNT(*)) OVER ()
        AS percentage_of_sales_lines,

    SUM(sales) AS total_sales,

    100.0 * SUM(sales) / SUM(SUM(sales)) OVER ()
        AS percentage_of_total_sales,

    SUM(profit) AS total_profit

FROM profitability_status

GROUP BY profit_status

ORDER BY
    CASE profit_status
        WHEN 'Profitable' THEN 1
        WHEN 'Break Even' THEN 2
        WHEN 'Loss' THEN 3
    END;
""").df().style.hide(axis="index").format({
    "sales_line_count": "{:,}",
    "percentage_of_sales_lines": "{:.2f}%",
    "total_sales": "${:,.2f}",
    "percentage_of_total_sales": "{:.2f}%",
    "total_profit": "${:,.2f}"
})

profit_status,sales_line_count,percentage_of_sales_lines,total_sales,percentage_of_total_sales,total_profit
Profitable,"8,058",80.63%,"$1,800,806.86",78.39%,"$442,528.31"
Break Even,65,0.65%,"$27,686.85",1.21%,$0.00
Loss,"1,871",18.72%,"$468,707.15",20.40%,"$-156,131.29"


**Results:**

Profitable sales lines accounted for **80.63%** of all sales lines and generated **$1,800,806.86**, or **78.39%** of total sales. These lines produced **$442,528.31** in gross profit.

Loss-generating sales lines represented **18.72%** of sales activity but accounted for **20.40%** of total revenue. Although these lines generated **$468,707.15** in sales, they produced a combined loss of **$156,131.29**. Break-even lines represented only **0.65%** of activity and generated **$27,686.85** in sales without contributing profit.

**Key Insights:**

Nearly one-fifth of sales lines generated a loss, demonstrating that substantial revenue did not translate into positive financial value. Loss-generating lines erased approximately **35.28%** of the gross profit produced by profitable lines, reducing the company's resulting net profit to approximately **$286,397**.

Loss-generating lines also represented a slightly larger share of revenue than of sales-line activity, indicating that losses were not confined to insignificant purchases. Reducing the frequency or severity of these losses could improve overall profitability without requiring additional sales growth. The next analysis identifies the category and discount combinations contributing to this problem.

### Loss-Generating Category and Discount Combinations

**Question:**

Which combinations of product category and discount level generate overall financial losses?

**Objective:**

Identify category and discount combinations with negative total profit and compare their sales-line activity, total sales, total loss, and profit margin. This analysis determines whether the financial effect of discounting differs across product categories and highlights the combinations requiring the greatest attention.

**Variables:**

- `category`
- `discount`
- `sales`
- `profit`

In [18]:
con.execute("""
SELECT
    category,
    100.0 * discount AS discount_percentage,
    COUNT(*) AS sales_line_count,
    SUM(sales) AS total_sales,
    SUM(profit) AS total_profit,

    100.0 * SUM(profit) / SUM(sales)
        AS profit_margin_percentage

FROM analysis_data

GROUP BY category, discount

HAVING SUM(profit) < 0

ORDER BY total_profit;
""").df().style.hide(axis="index").format({
    "discount_percentage": "{:.0f}%",
    "sales_line_count": "{:,}",
    "total_sales": "${:,.2f}",
    "total_profit": "${:,.2f}",
    "profit_margin_percentage": "{:.2f}%"
})

category,discount_percentage,sales_line_count,total_sales,total_profit,profit_margin_percentage
Office Supplies,80%,300,"$16,963.76","$-30,539.04",-180.03%
Technology,70%,23,"$15,601.51","$-19,579.32",-125.50%
Office Supplies,70%,380,"$22,559.39","$-16,601.10",-73.59%
Furniture,40%,75,"$45,614.41","$-16,187.40",-35.49%
Furniture,50%,54,"$20,983.47","$-12,871.20",-61.34%
Furniture,30%,222,"$99,470.35","$-10,695.32",-10.75%
Technology,50%,12,"$37,935.07","$-7,635.23",-20.13%
Technology,40%,131,"$70,803.38","$-6,869.65",-9.70%
Furniture,60%,138,"$6,644.70","$-5,944.66",-89.46%
Furniture,70%,15,"$2,459.38","$-3,894.94",-158.37%


**Results:**

The largest loss occurred among Office Supplies sold at an 80% discount. This combination generated only **$16,963.76** in sales but produced a **$30,539.04 loss**, resulting in a **-180.03% profit margin**.

Technology sold at a 70% discount generated the second-largest loss at **$19,579.32**, followed by Office Supplies at a 70% discount with a **$16,601.10 loss**. Within Furniture, the 40% discount level produced the largest loss at **$16,187.40**. Furniture also generated negative total profit at every displayed discount level from 30% through 70%.

**Key Insights:**

The discount level at which profitability became negative differed by product category. Furniture generated losses beginning at 30%, Technology generated losses at 40% and above, and Office Supplies appeared among the loss-generating combinations only at the extreme 70% and 80% discount levels. This suggests that product categories have different capacities to absorb discounts.

Furniture appears particularly sensitive to discounting, which helps explain the category's weak overall **2.49% profit margin** identified earlier. The most severe individual losses occurred at 70% and 80%, where discounts exceeded the available product margins. Rather than applying a single discount policy across all products, the business should establish category-specific discount limits based on historical profitability.

## Shipping & Time Analysis

Shipping and time analysis evaluates operational efficiency and temporal business trends. By analyzing fulfillment times, shipping methods, and sales performance across months and years, this section identifies seasonal patterns, monitors delivery performance, and uncovers trends that can support operational planning.

### Annual Business Performance

**Question:**

How did the company’s financial performance and order activity change from year to year?

**Objective:**

Evaluate annual performance by comparing order volume, total sales, total profit, average order value, and overall profit margin. Examining these measures together helps determine whether revenue changes were driven by the number of orders, the average value of each order, or both.

**Variables:**

- `order_year`
- `order_id`
- `sales`
- `profit`

In [29]:
con.execute("""
SELECT
    order_year,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(sales) AS total_sales,
    SUM(profit) AS total_profit,
    SUM(sales) / COUNT(DISTINCT order_id) AS avg_order_value,
    100.0 * SUM(profit) / SUM(sales) AS profit_margin_percentage
FROM analysis_data
GROUP BY order_year
ORDER BY order_year;
""").df().style.hide(axis="index").format({
    "total_orders": "{:,}",
    "total_sales": "${:,.2f}",
    "total_profit": "${:,.2f}",
    "avg_order_value": "${:,.2f}",
    "profit_margin_percentage": "{:.2f}%"
})

order_year,total_orders,total_sales,total_profit,avg_order_value,profit_margin_percentage
2014,969,"$484,247.50","$49,543.97",$499.74,10.23%
2015,"1,038","$470,532.51","$61,618.60",$453.31,13.10%
2016,"1,315","$609,205.60","$81,795.17",$463.27,13.43%
2017,"1,687","$733,215.26","$93,439.27",$434.63,12.74%


**Results:**

Annual order volume increased from **969 orders in 2014** to **1,687 orders in 2017**, representing overall growth of approximately **74.10%**. Total sales increased from **$484,247.50** to **$733,215.26**, while total profit rose from **$49,543.97** to **$93,439.27**.

Sales declined slightly to **$470,532.51** in 2015 despite an increase in order volume. Performance then strengthened in 2016 and 2017, with both years producing higher sales and profit than the preceding year. The highest profit margin occurred in 2016 at **13.43%**, while average order value was highest in 2014 at **$499.74** and lowest in 2017 at **$434.63**.

**Key Insights:**

The company experienced substantial overall growth, with both order volume and total profit increasing across the four-year period. From 2014 to 2017, profit grew by approximately **88.60%**, outpacing the approximately **51.41%** increase in sales.

However, average order value declined by approximately **13.03%** over the same period. This indicates that revenue growth was driven primarily by a larger number of orders rather than increased spending per order. Although 2017 generated the highest total sales and profit, its **12.74% profit margin** was lower than the 2016 peak, suggesting that the company should monitor whether continued order growth is translating into equally strong improvements in profitability.

### Monthly Sales and Profit Trends

**Question:**

Which months generate the highest sales and profit?

**Objective:**

Compare sales and profitability across calendar months to identify recurring seasonal patterns. This can help the business anticipate periods of higher demand and plan marketing, staffing, and inventory decisions.

**Variables:**

- `order_month`
- `sales`
- `profit`

In [26]:
con.execute("""
SELECT
    order_month,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(sales) AS total_sales,
    SUM(profit) AS total_profit,
    SUM(sales) / COUNT(DISTINCT order_id) AS avg_order_value,
    100.0 * SUM(profit) / SUM(sales) AS profit_margin_percentage
FROM analysis_data
GROUP BY order_month
ORDER BY order_month;
""").df().style.hide(axis="index").format({
    "total_orders": "{:,}",
    "total_sales": "${:,.2f}",
    "total_profit": "${:,.2f}",
    "avg_order_value": "${:,.2f}",
    "profit_margin_percentage": "{:.2f}%"
})

order_month,total_orders,total_sales,total_profit,avg_order_value,profit_margin_percentage
1,178,"$94,924.84","$9,134.45",$533.29,9.62%
2,162,"$59,751.25","$10,294.61",$368.83,17.23%
3,354,"$205,005.49","$28,594.69",$579.11,13.95%
4,343,"$137,762.13","$11,587.44",$401.64,8.41%
5,369,"$155,028.81","$22,411.31",$420.13,14.46%
6,364,"$152,718.68","$21,285.80",$419.56,13.94%
7,338,"$147,238.10","$13,832.66",$435.62,9.39%
8,341,"$159,044.06","$21,776.94",$466.40,13.69%
9,688,"$307,649.95","$36,857.48",$447.17,11.98%
10,417,"$200,322.98","$31,784.04",$480.39,15.87%


**Results:**

November generated the highest total sales at **$352,461.07** from **753 orders**, followed by December with **$325,293.50** from **702 orders** and September with **$307,649.95** from **688 orders**. December produced the highest total profit at **$43,369.19**, while September and November generated **$36,857.48** and **$35,468.43**, respectively.

February had the lowest sales and order volume, generating **$59,751.25** from **162 orders**. Despite its lower activity, February achieved the highest profit margin at **17.23%**. April produced the lowest profit margin at **8.41%**, while March recorded the highest average order value at **$579.11**.

**Key Insights:**

Business activity was concentrated in the later months of the year. September through December accounted for approximately **51.62% of total sales**, indicating that the fall and holiday period is especially important to overall performance. The company may benefit from preparing inventory, staffing, shipping capacity, and marketing campaigns in advance of this higher-demand period.

The highest-sales months were not always the most efficient at converting revenue into profit. November generated the most sales but achieved a profit margin of only **10.06%**, while lower-volume February produced the strongest margin at **17.23%**. This demonstrates the importance of evaluating sales alongside profitability rather than assuming that the highest-revenue months are automatically the strongest-performing months.

Because the results combine each calendar month across all four years, they indicate an overall seasonal pattern but do not establish whether the same pattern occurred consistently in every individual year.

### Fulfillment Performance by Shipping Mode

**Question:**

How does fulfillment performance vary across shipping modes?

**Objective:**

Compare order usage and fulfillment time across shipping modes to evaluate differences in delivery speed and determine whether the modes follow the expected service hierarchy. This analysis provides insight into shipping behavior while recognizing that formal service performance cannot be assessed without promised delivery standards or service-level targets.

**Variables:**

- `ship_mode`
- `order_id`
- `fulfillment_days`

In [25]:
con.execute("""
WITH order_summary AS (
    SELECT
        order_id,
        ship_mode,
        MAX(fulfillment_days) AS fulfillment_days
    FROM analysis_data
    GROUP BY order_id, ship_mode
)

SELECT
    ship_mode,
    COUNT(*) AS total_orders,
    AVG(fulfillment_days) AS avg_fulfillment_days,
    MIN(fulfillment_days) AS minimum_fulfillment_days,
    MAX(fulfillment_days) AS maximum_fulfillment_days
FROM order_summary
GROUP BY ship_mode
ORDER BY avg_fulfillment_days;
""").df().style.hide(axis="index").format({
    "total_orders": "{:,}",
    "avg_fulfillment_days": "{:.2f}",
    "minimum_fulfillment_days": "{:.0f}",
    "maximum_fulfillment_days": "{:.0f}"
})

ship_mode,total_orders,avg_fulfillment_days,minimum_fulfillment_days,maximum_fulfillment_days
Same Day,264,0.05,0,1
First Class,787,2.19,1,4
Second Class,964,3.23,1,5
Standard Class,"2,994",5.00,3,7


**Results:**

Standard Class was the most frequently used shipping mode, accounting for **2,994 orders**, followed by Second Class with **964 orders**, First Class with **787 orders**, and Same Day with **264 orders**.

Average fulfillment time increased across the shipping service levels. Same Day orders averaged **0.05 days**, First Class averaged **2.19 days**, Second Class averaged **3.23 days**, and Standard Class averaged **5.00 days**. Same Day orders were fulfilled within zero to one day, while Standard Class orders ranged from three to seven days.

**Key Insights:**

Standard Class represented approximately **59.77%** of all orders, indicating that most customers used the slowest available shipping option. In contrast, Same Day represented only approximately **5.27%** of orders.

The average fulfillment times followed the expected service hierarchy: Same Day was fastest, followed by First Class, Second Class, and Standard Class. This suggests that the shipping modes meaningfully differentiate orders by fulfillment speed. However, determining whether each mode met customer expectations would require access to the company’s promised delivery standards or service-level targets.

Reducing the data to one row per order ensured that orders containing multiple product lines did not receive greater weight in the fulfillment calculations.

## Overall Findings and Business Recommendations

The analysis identified substantial business growth alongside several opportunities to improve profitability. From 2014 to 2017, total sales increased by approximately **51.41%**, while profit increased by approximately **88.60%**. This growth was driven primarily by higher order volume, as average order value declined over the same period.

Customer performance varied considerably across segments and value tiers. Elite customers represented approximately **10% of the customer base** but contributed **30.88% of sales** and **42.66% of profit**, making them particularly important for retention efforts. Although Consumer customers generated the greatest total revenue, Corporate and Home Office demonstrated stronger average sales per customer or profit margins.

Geographic and product-classification results showed that high sales did not always translate into profitability. Texas generated the third-highest state sales but produced the largest state-level loss, while Furniture generated substantial revenue with a profit margin of only **2.49%**. Tables and Bookcases were major contributors to Furniture's weak performance.

Discounting emerged as one of the clearest profitability concerns. Aggregate profitability became negative at the **30% discount level**, and every sales line discounted by **45% or more** generated a loss. However, the effect differed across categories, supporting category-specific discount policies rather than a single company-wide threshold.

Based on these findings, the business should consider:

- Prioritizing retention efforts for Elite and Premium customers.
- Investigating the lower profit margins of the Consumer and High Value customer groups.
- Reviewing pricing, product mix, and discounting practices in loss-generating states.
- Establishing category-specific discount limits, with particular attention to Furniture.
- Preparing inventory, staffing, and shipping capacity for increased demand from September through December.
- Monitoring average order value alongside order growth to ensure that higher activity continues to produce sustainable financial gains.